# Edge-IIoTset Preprocessing Walk-through

FLEAD streams the [Edge-IIoTset](https://www.kaggle.com/datasets/sibasispradhan/edge-iiotset-dataset) dataset as 2,400 simulated IoT devices. The production pipeline runs headless:

1. `scripts/data_preprocessor.py` downloads the dataset, drops identifier columns, fits the preprocessing on a sample and transforms the 1.2 GB CSV in chunks.
2. `scripts/convert_chunks_to_device_csvs.py` assigns rows randomly to devices and writes `data/processed/device_N.csv`.

`START.bat` / `./start` run both automatically when `data/processed` is empty. This notebook reproduces the same steps **on a sample, using the same functions**, so each decision can be inspected. It does not overwrite the processed data.

Requires a Kaggle API token at `kaggle/kaggle.json` (mounted into the `jupyter-dev` container) if the raw data is not downloaded yet.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Repository root: works from notebooks/ locally and from /app/notebooks in the jupyter-dev container
ROOT = Path.cwd().resolve()
while not (ROOT / "scripts" / "data_preprocessor.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

import config_loader
import data_preprocessor as dp

print("Repository root:", ROOT)

## 1. Download

Skipped when `data/raw/DNN-EdgeIIoT-dataset.csv` already exists.

In [ ]:
csv_path = dp.DATA_RAW / dp.SOURCE_CSV
if not csv_path.exists():
    dp.ensure_kaggle_credentials()
    dp.download_edge_iiot()
print(f"{csv_path.name}: {csv_path.stat().st_size / 2**30:.2f} GB")

## 2. Sample the raw data

The raw file is ordered by traffic type (benign rows first, then each attack), so reading only the first rows would show almost no attacks. The sample takes 5% of every 100k-row chunk instead, the same way the preprocessor samples rows to fit its pipeline.

In [ ]:
parts = []
for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=dp.CHUNK_SIZE, low_memory=False)):
    parts.append(dp.prepare_chunk(chunk).sample(frac=0.05, random_state=i))
sample = pd.concat(parts, ignore_index=True)
print(f"Sample: {len(sample):,} rows x {sample.shape[1]} columns")
sample.head()

In [ ]:
label_col = dp.guess_label_column(list(sample.columns))
print("Label column:", label_col)
print(f"Attack share: {sample[label_col].mean():.1%}")
sample["Attack_type"].value_counts().plot.barh(figsize=(8, 5), title="Traffic types in the sample")
plt.tight_layout()

## 3. Columns removed before modelling

* **Identifiers, timestamps and free text** (`DROP_COLUMNS`): unique per packet, so they carry no pattern that generalises and would explode one-hot encoding. The Edge-IIoTset authors drop the same columns.
* **Every label-like column** (`LABEL_COLUMNS`): the dataset ships both `Attack_label` (0/1) and `Attack_type` (name). Keeping `Attack_type` as a feature would leak the answer into the model.

In [ ]:
print("Dropped identifier/text columns:", dp.DROP_COLUMNS)
print("Label columns excluded from features:", sorted(c for c in dp.LABEL_COLUMNS if c in sample.columns))

## 4. Feature types and preprocessing pipeline

A column is numeric when at least 95% of its non-missing values parse as numbers; this is decided once on the sample so every chunk is transformed the same way. Numeric features are mean-imputed and **standardized**, so feature *i* has the same scale on every device, which federated averaging of model weights relies on.

In [ ]:
features = sample.drop(columns=[c for c in dp.LABEL_COLUMNS if c in sample.columns]).dropna(axis=1, how="all")
numeric_cols, cat_cols = dp.split_feature_types(features)
print(f"{len(numeric_cols)} numeric, {len(cat_cols)} categorical feature columns")

pipeline = dp.build_pipeline(numeric_cols, cat_cols)
X = pipeline.fit_transform(dp.coerce_types(features[numeric_cols + cat_cols], numeric_cols, cat_cols))
names = list(pipeline.named_steps["pre"].get_feature_names_out())
transformed = pd.DataFrame(X.toarray() if hasattr(X, "toarray") else X, columns=names)
transformed.describe().T[["mean", "std", "min", "max"]].round(2).head(15)

## 5. Device files produced by the pipeline

The converter assigns rows to devices at random in equal shares, then shuffles each device's reading order. Assigning contiguous slices of the class-ordered file instead produced devices that were 100% benign or 100% attack.

In [ ]:
summary = json.loads((dp.DATA_PROCESSED / "processing_summary.json").read_text())
device_files = sorted(dp.DATA_PROCESSED.glob("device_*.csv"))
print(f"Rows processed: {summary['total_samples']:,} | features: {summary['n_features']} | device files: {len(device_files)}")

rng = np.random.default_rng(0)
picked = rng.choice(device_files, size=min(200, len(device_files)), replace=False)
stats = pd.DataFrame([
    {"device": p.stem, "rows": len(df), "attack_share": df["label"].mean()}
    for p in picked for df in [pd.read_csv(p, usecols=["label"])]
])
print(stats.describe().round(3))
stats["attack_share"].plot.hist(bins=20, title="Attack share per device (200 random devices)")
plt.xlabel("share of attack readings");

## 6. Streamed vs. held-out rows

The Kafka producer streams only the first `stream_rows_per_device` rows of each device; Spark evaluates the federated model on the remaining rows, which no device has trained on.

In [ ]:
split = config_loader.get_stream_config()
rows = int(stats["rows"].median())
print(f"Streamed rows per device: {split['stream_rows_per_device']} | held out: {rows - split['stream_rows_per_device']} (median device has {rows} rows)")